# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library. All references to dataset entities (record sets, fields, and columns) are by their unique `@id`.

### Dataset Source
Croissant schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL (reference by variable for reuse)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show the dataset metadata (name & description)
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
This section lists available record sets, their `@id`s, corresponding fields, and field `@id`s.

In [ ]:
# List record sets and their fields, referencing by @id.
record_sets = dataset.metadata.record_sets

print('Record sets available:')
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id} | Name: {rs.name}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id}\tName: {field.name}\tDataType: {field.data_type}")
    print()

## 3. Data Extraction
Load all record sets into pandas DataFrames. Use the record set and field `@id`s from above for all data access.

In [ ]:
# Build dictionary mapping record_set @id to DataFrame
dataframes = {}
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

# Extract data for each record set
for record_set_id in record_set_ids:
    df = pd.DataFrame(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = df
    print(f"Loaded record set @id: {record_set_id} with shape {df.shape}")
    if len(df.columns) < 20:
        print(f"Columns: {list(df.columns)}")
    else:
        print(f"Columns (first 20): {list(df.columns)[:20]}")
    print()

# Pick the first record set for further analysis
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id is not None:
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
We demonstrate basic filtering, normalization, and grouping on a numeric field, referencing by its field `@id`.

> **Tip:** Replace `your_numeric_field_id` and `your_group_field_id` below with the `@id`s for numeric/categorical fields listed above.

In [ ]:
### Example: Filter, normalize, and group by a field

# EDIT these variables according to available field @id's from the previous overview
# For demonstration, select the first numeric field @id in the first record set
main_fields = dataset.metadata.record_sets[0].fields if dataset.metadata.record_sets else []
numeric_field_id = None
categorical_field_id = None

for f in main_fields:
    # Heuristically pick first Float/Integer field for EDA, and a Text field for group
    if not numeric_field_id and f.data_type in ["schema:Float", "schema:Number", "schema:Integer"]:
        numeric_field_id = f.id
    if not categorical_field_id and f.data_type == "schema:Text":
        categorical_field_id = f.id

if numeric_field_id is not None:
    df = dataframes[main_record_set_id]
    # Filter records with the numeric value > threshold
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype.kind in 'fi' else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records in '{main_record_set_id}' with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df[[numeric_field_id]].head())

    # Normalize the field
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std if std else 0
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # (Optional) Group by a categorical/text field @id
    if categorical_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(categorical_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped average {numeric_field_id} by {categorical_field_id}:")
        display(grouped_df.head())
else:
    print('No numeric field detected for EDA. Please check the field types and update the variables.')

## 5. Visualization
We visualize the distribution of the selected numeric field (`@id: {}`), and a grouped barplot by the selected categorical field (`@id: {}`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if numeric_field_id is not None and main_record_set_id is not None:
    df = dataframes[main_record_set_id]

    # Histogram
    plt.figure(figsize=(6,4))
    df[numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouped_df exists, barplot by group
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(8,4))
        sns.barplot(x=categorical_field_id, y=numeric_field_id, data=grouped_df)
        plt.title(f"Average {numeric_field_id} by {categorical_field_id}")
        plt.xticks(rotation=30, ha='right')
        plt.show()

## 6. Conclusion
- This notebook demonstrated how to explore the FAIR^2 dataset using the `mlcroissant` library, referencing all data structures by their Croissant `@id`.
- You can repeat analysis for other record sets or fields by substituting their `@id`s above.

#### Next steps
Use the Croissant metadata (`dataset.metadata`) to inspect further details on provenance, licensing, and field semantics as needed for your analysis.